# RAG 평가 개요
- RAG 평가란 RAG 시스템이 주어진 입력에 대해 얼마나 효과적으로 관련 정보를 검색하고, 이를 기반으로 정확하고 유의미한 응답을 생성하는지를 측정하는 과정이다. 
- **평가 요소**
    - **검색 단계 평가**
        - 입력 질문에 대해 검색된 문서나 정보의 관련성과 정확성을 평가.
    - **생성 단계 평가**
        - 검색된 정보를 기반으로 생성된 응답의 품질, 정확성등을 평가.
- **평가 방법**
    - **온/오프라인 평가**
        1. **오프라인 평가**
            - 미리 준비된 데이터셋을 활용하여 RAG 시스템의 성능을 측정한다.
        2. **온라인 평가**
            - 실제 사용자 트래픽과 피드백을 기반으로 시스템의 실시간 성능을 평가한다.
    - **정량적/정성적 평가**
        1. 정량적 평가
            - 자동화된 지표를 사용하여 생성된 텍스트의 품질을 평가한다.
        2. 정성적 평가
            - 전문가나 일반 사용자가 직접 생성된 응답의 품질을 평가하여 주관적인 지표를 평가한다.

# [RAGAS](https://www.ragas.io/)
- RAGAS는 RAG 파이프라인을 **정량적으로 평가하는** 오픈소스 프레임 워크이다. 
- RAGAS 문서: https://docs.ragas.io/en/stable/
## 설치
- `uv pip install ragas rapidfuzz`

## RAGAS 평가 지표 개요
![ragas_score](figures/ragas_score.png)
- **Generation**
    - llm 모델이 생성한 답변에 대한 평가 지표들.
    - **Faithfulness(신뢰성)**
        -  생성된 답변과 검색된 문서(context)간의 관련성을 평가하는 지표
        -  생성된 답변이 주어진 문맥(context)에 얼마나 충실한지를 평가하는 지표로 할루시네이션에 대한 평가로 볼 수있다.
    - **Answer relevancy(답변 적합성)**
        - 생성된 답변과 사용자의 질문간의 관련성을 평가하는 지표
        - 생성된 답변이 사용자의 질문과 얼마나 관련성이 있는지를 평가하는 지표.
- **Retrieval**
    -  질문에 대해 검색한 문서(context)들에 대한 평가
    -  **Context Precision(문맥 정밀도)**
        -  검색된 문서(context)들 중 질문과 관련 있는 것들이 **얼마나 상위 순위에 위치하는지** 평가하는 지표.
    -  **Context Recall(문맥 재현률)**
        -  검색된 문서(context)가 정답(ground-truth)의 정보를 얼마나 포함하고 있는지 평가하는 지표.
- 이러한 지표들은 RAG 파이프라인의 성능을 다각도로 평가하는 데 활용된다.
![RAGAS_score2](figures/RAGAS_score2.png)

## 주요 평가지표
### Generation 평가
- LLM이 생성한 답변에 대한 평가
  
#### Faithfulness (신뢰성)
- 생성된 답변이 얼마나 주어진 검색 문서들(context)를 잘 반영해서 생성되었는지 평가한다. 할루시네이션에 대한 평가라고 할 수 있다. 
- 점수범위: **0 ~ 1** (1에 가까울수록 좋음)
- 답변에 포함된 모든 주장이 context에서 얼마나 추출 가능한지를 확인한다.

##### 평가 방법
1. Answer에서 주장 구문(claim statement)들을 생성(추출)한다. (주장이란, 질문(user input)과 관련된 내용)
    - 예) 
        - **질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요? 
        - **LLM 답변**: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 3000만명이다.
2. 각 주장들을 context로 부터 추론 가능한지 판단한다. 이를 바탕으로 faithfulness 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다. .... 한국의 인구는 5000만명이고 서울에 1000만이 살고 있다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 추론가능한 주장.
            - 한국의 인구는 3000만명이다. -> context에서 추론 불가능한 주장.
3. **Faithfulness score** 를 계산한다. 총 주장 수 중에서 context로 부터 추론가능한 주장의 개수.    
    - 예)
        - Faithfulness Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 유추할 수있다.)
    - LLM 답변에서 주장을 추출 하는 것과 각 주장이 context에서 추론 가능한 지를 판단하는 것은 LLM 을 활용한다.
- 공식
    $$
    \text{Faithfulness Score}\;=\;\cfrac{\text{주어진\;context\;에서\;추론할\;수\;있는\;주장의\;개수}}{\text{총\;주장\;개수}}
    $$

### Answer relevancy (답변 적합성)
- 생성된 답변이 질문(user input)에 얼마나 잘 부합하는 지를 평가한다.
- 점수 범위: -1~1 (1에 가까울수록 좋음)
- LLM이 생성한 답변을 기반으로 질문들을 생성한다. 이렇게 생성한 질문들과 실제 질문(user input) 간의 유사도를 측정한다.

#### 평가방법
1. LLM이 생성한 답변을 기반으로 질문들을 생성한다.
    - 예) 
        - **LLM** 답변: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
2. 실제 질문과 생성한 질문간의 코사인 유사도를 측정한다. 그 평균이 최종 점수가 된다.
    - 예)
        - **실제 질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요?
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
- 공식
  $$
    \cfrac{1}{N} \sum_{i=1}^{N} \text{cosine\_similarity}(q_{\text{user}_{_i}}, q_{\text{generated}})
  $$

## Retrieval 평가
Vector store에서 검색한 context에 대한 평가

### Context Precision
- 검색된 문서(context)들 중 질문과 관련 있는 것들이 얼마나 **상위 순위**에 있는 지 평가.
- 점수 범위: 0~1 (1에 가까울수록 좋음)


#### 평가방법

- 공식
$$
 \text{Context\;Precision@K} = \frac{\sum_{k=1}^{K} \left( \text{Precision@k} \times v_k \right)}{\ 상위\;K개\;결과에서의\;관련\;항목\;수}
$$
$$
 \text{Precision@k} = \frac{\text{True\;positive@k}}{(\text{True\;positive@k} + \text{False\;positive@k})} \\
$$
- $\text{Precision@k}$: 개별 문서에 대한 Precision
- K: context 의 개수(chuck 수)
- $v_k$: 관련성 여부로 0 또는 1. (0: 관련 없음, 1: 관련 있음)

#### 예시
- 질문과 context 관련성의 예
    - 질문: 한국의 수도는 어디이고 인구는 얼마나 되나요?
    - **높은 정밀도 context들**: 질문과 직접적인 관련이 있는 문서들
        - 한국의 수도는 서울이고 인구는 5000만명 입니다. 
        - 한국의 수도는 서울입니다.
        - 한국은 동아시아에 위치해 있는 국가로 수도는 서울입니다.
        - 한국의 인구는 5000만명 입니다.
    - **낮은 정밀도 context**: 한국과 관련있어 검색될 수 있지만 질문과 직접적 관련이 없다. 
        - 한국은 동아시아에 위치한 국가입니다.
        - 한국의 K-pop은 전 세계적으로 유명합니다.
        - 비빔밥, 불고기는 한국의 대표적인 음식입니다.
    - **높은 정밀도의 context이 상위 순위에 위치했으면 높은 점수를 받는다.**

- 점수 계산 예:
    - **상위 5개의 검색 결과 중 1번째, 3번째, 4번째 문서가 관련이 있다고 가정하자.**
    - **Precision@K 계산**
        ```bash
            Precision@1 = 1/1 = 1.0    # True positive@1/(True positive@1 + False positive@1).  1/1(1번 문서 계산 시에는 1개 문서만 있으므로 분모가 1이 된다.)
            Precision@2 = 1/2 = 0.5
            Precision@3 = 2/3 ≈ 0.67    
            Precision@4 = 3/4 = 0.75
            Precision@5 = 3/5 = 0.6
        ```
    - **vk의 값**
        - 1번째: $v_1 = 1$ - 관련있음
        - 2번째: $v_2 = 0$ - 관련없음
        - 3번째: $v_3 = 1$ - 관련있음
        - 4번째: $v_4 = 1$ - 관련있음
        - 5번째: $v_5 = 0$ - 관련없음

    - **Context Precision@5**
        $$
        \text{Context\;Precision@5} = \frac{(1.0 \times 1) + (0.5 \times 0) + (0.67 \times 1) + (0.75 \times 1) + (0.6 \times 0)}{3} = \frac{1.0 + 0 + 0.67 + 0.75 + 0}{3} ≈ 0.807
        $$

### Context Recall (문맥 재현률)
- 검색된 문서(context)가 얼마나 정답(ground-truth)의 정보를 포함있는 지 평가하는 지표
- 점수 범위: 0~1 (1에 가까울수록 좋음)
- **정답(ground truth)의 각 주장(claim)이 검색된 context와 얼마나 일치**하는지 계산함.

#### 평가방법
1. 정답에서 주장(claim)들을 생성(추출)한다.
    - 예) 
        - **정답**: 한국의 수도는 서울이고 인구수는 5000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 5000만명이다.
2. 각 주장(claim)의 정보를 검색된 contexts에서 찾을 수 있는지 판별한다. 이를 바탕으로 context recall 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 찾을 수 있다.
            - 한국의 인구는 5000만명이다. -> context에서 찾을 수 없다.
3. **Context Recall Score** 를 계산한다. 총 주장 수 중에서 context로 부터 찾을 수 있는 주장의 개수.
    - 예)
        - Context Recall Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 찾을 수 있다.)

- 공식
    $$
    \text{Context Recall Score}\;=\;\cfrac{\text{GT의\;주장\;중\;주어진\;context\;에서\;찾을\;수\;있는\;주장의\;개수}}{\text{GT의\;총\;주장\;개수}}
    $$ 

# RAGAS 평가 실습

In [2]:
# !uv pip install ragas rapidfuzz
# !uv pip install langchain_neo4j langchain_experimental neo4j
# 설치 후 커널 재시작


In [ ]:
# docker run \
#     --name neo4j \
#     -p 7474:7474 -p 7687:7687 \
#     -d \
#     -e NEO4J_AUTH=neo4j/password \
#     -e NEO4J_PLUGINS='["apoc"]' \
#     neo4j:latest
# 브라우저 콘솔: http://localhost:7474 (neo4j / password 로 로그인)


In [ ]:
# !uv pip install langchain_experimental --break-system-packages

Resolved 48 packages in 766ms
Prepared 1 package in 273ms
Installed 1 package in 75ms
 + langchain-experimental==0.4.2


In [3]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_experimental.graph_transformers import LLMGraphTransformer

from dotenv import load_dotenv

load_dotenv()


C:\Users\Playdata\AppData\Local\Temp\ipykernel_8700\1426098559.py:8: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


True

In [12]:
# ##############################################################
# 데이터 준비 (Neo4j 전용 원본 문서)
# ------------------------------------------------------------
# Qdrant와 Neo4j는 입력 문서가 서로 다른 별도의 소스입니다.
# 아래 함수는 Neo4j 지식그래프 구축에만 쓰이는 문서를 로드/분할합니다.
# file_path를 실제 Neo4j용 원본 문서 경로로 바꿔서 사용하세요.
##############################################################

def load_and_split_neo4j_docs(file_path="data/olympic_wiki.md"):
    """
    Neo4j 지식그래프 구축에 사용할 원본 문서를 읽어서 청크로 분할한다.
    (Qdrant 쪽 문서와는 별개의 소스 파일이다.)
    """
    with open(file_path, "r", encoding="utf-8") as fr:
        raw_text = fr.read()

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "H1"),
            ("##", "H2"),
            ("###", "H3"),
        ],
    )

    return splitter.split_text(raw_text)


In [13]:
#################################################################
# Neo4j 연결
# 문서 -> 지식그래프(엔티티/관계) 구축
#################################################################

import os
from langchain_neo4j import Neo4jGraph


def get_graph() -> Neo4jGraph:
    """
    .env 파일의 NEO4J_URI / NEO4J_USERNAME / NEO4J_PASSWORD / NEO4J_DATABASE
    환경변수를 읽어서 Neo4j에 연결한다.
    """
    return Neo4jGraph(
        url=os.getenv("NEO4J_URI"),
        username=os.getenv("NEO4J_USERNAME"),
        password=os.getenv("NEO4J_PASSWORD"),
        database=os.getenv("NEO4J_DATABASE", "neo4j"),
    )


def build_knowledge_graph(graph: Neo4jGraph, documents, clear_existing: bool = True) -> Neo4jGraph:
    """
    문서(list[Document])에서 LLM으로 엔티티/관계를 추출해 Neo4j에 지식그래프로 적재한다.

    Args:
        graph: 연결된 Neo4jGraph 객체
        documents: load_and_split_neo4j_docs()로 만든 청크 문서들 (Neo4j 전용 원본)
        clear_existing: 재실행 시 기존 그래프를 지우고 새로 적재할지 여부
    """
    if clear_existing:
        graph.query("MATCH (n) DETACH DELETE n")  # 기존 노드/관계 전부 삭제 (재실행 대비)

    extraction_llm = ChatOpenAI(model="gpt-4.1", temperature=0)
    llm_transformer = LLMGraphTransformer(llm=extraction_llm)

    ######################################
    # 문서 -> 그래프 문서(노드/관계) 변환
    ######################################
    graph_documents = llm_transformer.convert_to_graph_documents(documents)

    graph.add_graph_documents(
        graph_documents,
        baseEntityLabel=True,  # 모든 엔티티 노드에 공통 __Entity__ 라벨 부여(검색 편의)
        include_source=True,   # 원본 청크(Document) 노드도 함께 저장해 출처 추적 가능
    )

    graph.refresh_schema()  # Cypher 생성시 참고할 스키마 정보 갱신
    return graph


def get_retriever_chain(graph: Neo4jGraph, llm: ChatOpenAI, top_k: int = 5) -> GraphCypherQAChain:
    """
    자연어 질문 -> Cypher 쿼리 생성 -> Neo4j 실행 -> 자연어 응답 생성을 수행하는 체인.
    """
    cypher_chain = GraphCypherQAChain.from_llm(
        llm=llm,
        graph=graph,
        verbose=True,
        return_intermediate_steps=True,  # Cypher 조회 결과(context) 확인을 위해 필수
        allow_dangerous_requests=True,   # Cypher 실행을 허용하는 플래그 (필수)
        top_k=top_k,
    )
    return cypher_chain


In [14]:
# Neo4j 전용 원본 문서를 로드 (Qdrant 문서와는 별개의 파일)
documents = load_and_split_neo4j_docs(file_path="data/olympic_wiki.md")

graph = get_graph()
build_knowledge_graph(graph, documents)

print(f"청크(원본 문서) 개수: {len(documents)}")
print(graph.schema)


청크(원본 문서) 개수: 25
Node properties:
Person {id: STRING}
Product {id: STRING}
Event {id: STRING}
Organization {id: STRING}
Concept {id: STRING}
Document {id: STRING, text: STRING, H1: STRING, H2: STRING, H3: STRING}
Name {id: STRING}
Season {id: STRING}
Activity {id: STRING}
Place {id: STRING}
Date {id: STRING}
Award {id: STRING}
Number {id: STRING}
Symbol {id: STRING}
Artifact {id: STRING}
Country {id: STRING}
Creative_work {id: STRING}
Sport {id: STRING}
Thing {id: STRING}
Duration {id: STRING}
Age_group {id: STRING}
Group {id: STRING}
Location {id: STRING}
Language {id: STRING}
Media {id: STRING}
Program {id: STRING}
Object {id: STRING}
Value {id: STRING}
Time {id: STRING}
Phrase {id: STRING}
Amount {id: STRING}
Criterion {id: STRING}
Region {id: STRING}
Substance {id: STRING}
Rank {id: STRING}
Material {id: STRING}
Relationship properties:

The relationships:
(:Person)-[:FOUNDED]->(:Organization)
(:Person)-[:FOUNDED]->(:Event)
(:Person)-[:INSPIRED_BY]->(:Event)
(:Person)-[:FOUNDED_ON]

In [15]:
################################################################################
# 평가할 RAG Chain (GraphCypherQAChain 기반)
################################################################################

model = ChatOpenAI(model="gpt-4.1")  # Cypher 생성/응답 생성에 사용할 LLM
# GPT-5/o-series reasoning 모델을 쓰려면 temperature 파라미터를 넘기지 않아야 한다.

cypher_chain = get_retriever_chain(graph, model)

def format_context_from_steps(intermediate_steps: list) -> "list[str]":
    """
    GraphCypherQAChain의 intermediate_steps에서 Cypher 조회 결과를
    RAGAS가 요구하는 list[str] 형태(문장형 context)로 직렬화.
    intermediate_steps 구조: [{'query': cypher쿼리}, {'context': [{col: val, ...}, ...]}]
    """
    context_rows = intermediate_steps[1]["context"]
    return [str(row) for row in context_rows] if context_rows else []


def run_graph_rag(query: str) -> dict:
    result = cypher_chain.invoke({"query": query})
    return {
        "response": result["result"],                                              # LLM 응답
        "retrieved_context": format_context_from_steps(result["intermediate_steps"]),  # 검색한 context
    }

# 기존 Qdrant 버전 코드(chain.invoke(...))와 호환되도록 Runnable로 감싼다.
chain = RunnableLambda(run_graph_rag)


In [16]:
res = chain.invoke("올림픽 1회 개최지는 어디였나요?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (e:Event)-[:첫번째]->(p:Place) RETURN p.id
Full Context:
[{'p.id': '그리스'}]

> Finished chain.


In [17]:
# res = chain.invoke("1회 올림픽은 언제 어디서 열렸지")

In [18]:
print(res.keys())

dict_keys(['response', 'retrieved_context'])


In [19]:
res['response']

'올림픽 1회 개최지는 그리스입니다.'

In [20]:
res["retrieved_context"]

["{'p.id': '그리스'}"]

In [21]:
res

{'response': '올림픽 1회 개최지는 그리스입니다.', 'retrieved_context': ["{'p.id': '그리스'}"]}

# RAGAS 를 이용해 평가를 위한 합성 데이터 셋 만들기

- 평가 데이터셋 구성
  - `user_input`: 사용자 질문
  - `retrieved_contexts`: Neo4j(그래프DB)에서 Cypher로 검색한 context
  - `response`: LLM의 응답
  - `reference`: 정답

## TestsetGenerator
- **문서(retrieved_contexts)를 기준**으로 **질문**, **정답** 을 생성한다.
- 평가할 LLM으로 생성된 질문을 넣어 답변을 추출하여 데이터셋을 구성한다.

> **주의**
> - TestsetGenerator import 시 `No Module named langchain_community.chat_models.vertexai` Error 발생 
> - RAGAS와 langchain-community의 버전 호환성 문제 때문에 발생한다.
> - 해결
>   1. langchain_google_vertexai 설치
>       - `!uv pip install langchain_google_vertexai`
>   2. `.venv\Lib\site-packages\langchain_community\chat_models` 디렉토리 아래 `vertexai.py` 파일을 만들고 아래 코드를 넣는다.
>    ```python
>       try:
>           from langchain_google_vertexai import ChatVertexAI
>       except ImportError:
>           class ChatVertexAI:
>               def __init__(self, *args, **kwargs):
>                   raise ImportError(
>                       "ChatVertexAI requires langchain-google-vertexai. "
>                       "Install with: pip install langchain-google-vertexai"
>                   )
>    ```

In [22]:
!uv pip install langchain_google_vertexai

Checked 1 package in 211ms


In [23]:
# 주피터노트북 환경에서 비동기적 처리 위해
# script(.py) 로 작성할 경우는 필요 없다.

import nest_asyncio
nest_asyncio.apply()

In [24]:
################################
# testset -> Context들(문서들) - [질문 - 정답답변 + (GraphCypherQAChain이 찾은 context+LLM 응답: Chain 생성)]
# 1. Context(문서들)을 추출 -TestsetGenerator-> 질문과 정답답변 생성.

import random

# 데이터셋을 생성할 때 사용할 Context 를 추출.
# Neo4j 그래프를 만들 때 사용한 원본 청크 문서(documents)에서 그대로 샘플링한다.
# (TestsetGenerator는 그래프DB를 직접 조회하지 않고, 그래프 구축에 쓰인 원본 텍스트를 입력으로 받는다.)
total_docs = len(documents)

# 전체 문서 중에서 K(5)개만 랜덤 sampling
sample_docs = random.sample(documents, 5)

# Document.page_content만 추출해서 list[str]
docs = [doc.page_content for doc in sample_docs]


In [25]:
total_docs
sample_docs

[Document(metadata={'H1': '올림픽', 'H2': '근대 올림픽', 'H3': '동계 올림픽', 'id': '2ccb795ecab68856ac12655c197fbe55'}, page_content='동계 올림픽은 눈과 얼음을 이용하는 스포츠들을 모아 이루어졌으며 하계 올림픽 때 실행하기 불가능한 종목들로 구성되어 있다. 피겨스케이팅, 아이스하키는 각각 1908년과 1920년에 하계올림픽 종목으로 들어가 있었다. IOC는 다른 동계 스포츠로 구성된 새로운 대회를 만들고 싶어 했고, 로잔에서 열린 1921년 올림픽 의회에서 겨울판 올림픽을 열기로 합의했다. 1회 동계올림픽은 1924년, 프랑스의 샤모니에서 11일간 진행되었고, 16개 종목의 경기가 치러졌다. IOC는 동계 올림픽이 4년 주기로 하계 올림픽과 같은 년도에 열리도록 했다. 이 전통은 프랑스의 알베르빌에서 열린 1992년 올림픽 때까지 지속되었으나, 노르웨이의 릴레함메르에서 열린 1994년 올림픽부터 동계 올림픽은 하계 올림픽이 끝난지 2년후에 개최하였다.'),
 Document(metadata={'H1': '올림픽', 'H2': '우승자와 메달리스트', 'id': '752004b28981002abeba61e84f4173d7'}, page_content='개인 혹은 팀으로 경기에 출전해서 1위, 2위, 3위를 한 선수는 메달을 받는다. 1912년까지는 우승자에게 순금으로 된 금메달을 주었으며 그 후에는 도금된 금메달을 준다. 하지만, 2010 동계 올림픽에서는 전자제품 부속품을 녹여서 넣었다. 이러한 경우처럼 순금 외에 다른 물질을 넣을 경우에는 순금이 반드시 6g 이상을 함유하고 있어야 한다. 2위를 한 선수는 은메달을, 3위를 한 선수는 동메달을 받는다. 토너먼트로 진행되는 종목의 경우에는(복싱, 태권도 등) 3위를 구분하지 않고 준결승에서 패해서 3/4위전으로 간 선수들에게 모두 동메달을 수여한다. 1896년 하계 올림픽에서는 메달이 2개만 수여됐는데 1위에게 은메달을 주었고 

In [26]:
docs

['동계 올림픽은 눈과 얼음을 이용하는 스포츠들을 모아 이루어졌으며 하계 올림픽 때 실행하기 불가능한 종목들로 구성되어 있다. 피겨스케이팅, 아이스하키는 각각 1908년과 1920년에 하계올림픽 종목으로 들어가 있었다. IOC는 다른 동계 스포츠로 구성된 새로운 대회를 만들고 싶어 했고, 로잔에서 열린 1921년 올림픽 의회에서 겨울판 올림픽을 열기로 합의했다. 1회 동계올림픽은 1924년, 프랑스의 샤모니에서 11일간 진행되었고, 16개 종목의 경기가 치러졌다. IOC는 동계 올림픽이 4년 주기로 하계 올림픽과 같은 년도에 열리도록 했다. 이 전통은 프랑스의 알베르빌에서 열린 1992년 올림픽 때까지 지속되었으나, 노르웨이의 릴레함메르에서 열린 1994년 올림픽부터 동계 올림픽은 하계 올림픽이 끝난지 2년후에 개최하였다.',
 '개인 혹은 팀으로 경기에 출전해서 1위, 2위, 3위를 한 선수는 메달을 받는다. 1912년까지는 우승자에게 순금으로 된 금메달을 주었으며 그 후에는 도금된 금메달을 준다. 하지만, 2010 동계 올림픽에서는 전자제품 부속품을 녹여서 넣었다. 이러한 경우처럼 순금 외에 다른 물질을 넣을 경우에는 순금이 반드시 6g 이상을 함유하고 있어야 한다. 2위를 한 선수는 은메달을, 3위를 한 선수는 동메달을 받는다. 토너먼트로 진행되는 종목의 경우에는(복싱, 태권도 등) 3위를 구분하지 않고 준결승에서 패해서 3/4위전으로 간 선수들에게 모두 동메달을 수여한다. 1896년 하계 올림픽에서는 메달이 2개만 수여됐는데 1위에게 은메달을 주었고 2위에게 동메달을 주었다. 이때 3위에게는 아무것도 없었다. 현재의 메달 수여 방식은 1904년 하계 올림픽 때부터 시작되었다. 1948년부터는 4, 5, 6위를 한 선수에게는 인증서를 수여했다. 1984년 대회부터는 7, 8위를 한 선수에게도 인증서를 수여했다. 아테네에서 열린 2004년 하계 올림픽 때는 1, 2, 3위 선수에게 메달과 함께 올리브 화환도 같이 수여했다. 국가 올림픽 위원회(NOC)와 방

In [27]:
# 테스트 셋을 생성
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 참고: RAGAS 0.3.x 초반 버전에서는 gpt-5/o-series 모델 사용 시 temperature 파라미터
# 충돌로 에러가 났었지만, 이후 버전(0.4+)에서는 GPT-5/o-series를 공식 지원하도록 수정됐다.
# 에러가 나면 `ragas.__version__`을 확인하고 최신 버전으로 업그레이드할 것.
## Langchain의 LLM  모델과 Embedding 모델 -> RAGAS에서 사용할 수 있도록 변환(Wrapping)
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    llm_context="""
- 사람들이 올림픽에 대해서 궁금해 할 만한 질문들을 생성한다.
- 데이터셋은 반드시 한국어로 작성한다.
- 데이터셋은 JSON 문법을 지켜서 작성한다. 특히 구두점은 꼭 지켜야 한다.
- 생성된 내용이나 Document에 JSON문법에 맞지 않는 표현이 있으면 반드시 수정해서 처리한다.
""" # 질문/답변을 생성할 때 LLM에게 전달할 System Prompt 를 설정.
)


C:\Users\Playdata\AppData\Local\Temp\ipykernel_8700\1701836223.py:12: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_8700\1701836223.py:13: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


In [28]:
testset = generator.generate_with_chunks(
    docs, testset_size=10, # Conext 내용, 테스트셋 개수 (질문-답변 개수)
)

Applying SummaryExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

Node f7f91e9e-8142-4bb6-a463-83aa42e157df does not have a summary. Skipping filtering.
Node 7dbaede9-ad1a-432b-bf8e-540e3fa7ae99 does not have a summary. Skipping filtering.
Node e4c61cc7-4fc9-49f4-af95-c0a6152b3ce0 does not have a summary. Skipping filtering.
Node 6cb0196e-90d5-43ff-ad7b-c664fc8e4644 does not have a summary. Skipping filtering.
Node 372eccba-0572-4c0a-9d6e-1a24ef9d76d0 does not have a summary. Skipping filtering.


Applying EmbeddingExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [29]:
testset

Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input='프랑스에서 열린 동계 올림픽은 언제 개최되었나요?', retrieved_contexts=None, reference_contexts=['동계 올림픽은 눈과 얼음을 이용하는 스포츠들을 모아 이루어졌으며 하계 올림픽 때 실행하기 불가능한 종목들로 구성되어 있다. 피겨스케이팅, 아이스하키는 각각 1908년과 1920년에 하계올림픽 종목으로 들어가 있었다. IOC는 다른 동계 스포츠로 구성된 새로운 대회를 만들고 싶어 했고, 로잔에서 열린 1921년 올림픽 의회에서 겨울판 올림픽을 열기로 합의했다. 1회 동계올림픽은 1924년, 프랑스의 샤모니에서 11일간 진행되었고, 16개 종목의 경기가 치러졌다. IOC는 동계 올림픽이 4년 주기로 하계 올림픽과 같은 년도에 열리도록 했다. 이 전통은 프랑스의 알베르빌에서 열린 1992년 올림픽 때까지 지속되었으나, 노르웨이의 릴레함메르에서 열린 1994년 올림픽부터 동계 올림픽은 하계 올림픽이 끝난지 2년후에 개최하였다.'], retrieved_context_ids=None, reference_context_ids=None, response=None, multi_responses=None, reference='1회 동계올림픽은 1924년 프랑스의 샤모니에서 11일간 진행되었고, 1992년에는 프랑스의 알베르빌에서 올림픽이 열렸습니다.', rubrics=None, persona_name='Olympic History Enthusiast', query_style='PERFECT_GRAMMAR', query_length='SHORT'), synthesizer_name='single_hop_specific_query_synthesizer'), TestsetSample(eval_sample=SingleTurnSample(user_input='1948년부터 올림픽에서 몇 위 선수에게 인증서를 수여하기 시작했나요?

In [30]:
sample1 = testset.samples[5].eval_sample # 10개중 첫번째 테스트데이터
print("사용자질문:", sample1.user_input)
print("Context:", sample1.reference_contexts)
# 질문과 답변을 만들 때 사용한 context (검색시 찾아야 하는 문서)
print("생성된답변(정답):", sample1.reference)

##### 평가 대상 RAG system을 이용해서 채워 넣어야 한다.
print("평가대상 RAG의 답변:", sample1.response)
print("평가대상 RAG가 검색한 Context:", sample1.retrieved_contexts)

사용자질문: 1920년 하계 올림픽에서 개막식과 폐막식의 주요 행사들이 어떻게 정립되었으며, 이 대회가 이후 올림픽 의식의 전통에 어떤 영향을 미쳤는지 설명해 주세요.
Context: ["<1-hop>\n\n올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다.  \n#### 개막식  \n개막식 때는 올림픽 헌장에 따라 다양한 행사가 열린다. 개막식의 기본 토대는 벨기에 안트베르펜에서 열린 1920년 하계 올림픽 때 만들어졌다. 개막식은 대개 개최국의 국기가 게양되고 국가가 울려퍼지며 시작된다. 그 후에 개최국이 준비한 그들의 문화를 대표하는 음악, 춤, 영상 따위가 공연된다. 개막식은 아름답기가 매회가 지날수록 웅대해지고 복잡해지는데, 이는 전 대회보다 사람들의 기억에 오래도록 남기기 위함이다. 보도에 의하면 2008 베이징 올림픽 개막식 때 든 비용은 1억 달러로 그 대부분이 예술적인 부분에 들었다고 한다.  \n행사가 끝나면 다음에는 각국의 선수단이 입장한다. 올림픽의 발상지라는 영예를 가진 그리스가 전통적으로 맨 처음에 입장한다. 나머지 각국 선수단은 주최국에서 선택한 언어의 사전 순으로 입장하고 나서, 개최국 선수단이 제일 마지막에 입장한다. 그리스 아테네에서 열린 2004년 하계 올림픽에서는 그리스 국기가 맨 처음에 입장하고, 그리스 선수단이 맨 마지막에 입장했다. 개막식 마지막에는 올림픽 성화가 들어오고 마지막 성화 봉송자에게 전달할 때까지 돌게 된다. 마지막 성화 봉송자는 대체로 유명하고 올림픽에서 성공한 개최국의 선수가 하며 올림픽 성화를 경기장 내에 점화한다. 그 다음 올림픽조직위원장과 IOC 위원장이 개막 선언을 하는데 공식적으로 개막되었다는 것과 올림픽 성화가 점화되는 것을 선언한다. 그 다음에 올림픽조직위원장과 IOC 위원장이 개회사를 낭독하게 된다. 마자막으로 올림픽기 게양에 이어 올림픽 선서를 끝으로 개막식 과정은 모두 끝나게 된다. 2020년 하계 올림픽 이후로는 그리스 - 난민 올림픽 선수단 - 나머지 각국 선수

In [31]:
# 생성된 Testset을 Pandas DataFrame으로 변환

eval_df = testset.to_pandas()
eval_df.shape

(10, 7)

In [32]:
eval_df.head(3)

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,프랑스에서 열린 동계 올림픽은 언제 개최되었나요?,[동계 올림픽은 눈과 얼음을 이용하는 스포츠들을 모아 이루어졌으며 하계 올림픽 때 ...,"1회 동계올림픽은 1924년 프랑스의 샤모니에서 11일간 진행되었고, 1992년에는...",Olympic History Enthusiast,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
1,1948년부터 올림픽에서 몇 위 선수에게 인증서를 수여하기 시작했나요?,"[개인 혹은 팀으로 경기에 출전해서 1위, 2위, 3위를 한 선수는 메달을 받는다....","1948년부터는 4, 5, 6위를 한 선수에게는 인증서를 수여했다.",Olympic Sports Historian,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
2,헬싱키 올림픽에서 소련이 처음 참가한 배경과 그 이후 소련이 올림픽에서 보여준 성과...,[쿠베르탱이 말했던 원래 이념과는 반대로 올림픽이 정치 혹은 체제 선전의 장으로 이...,소련은 헬싱키에서 열린 1952년 하계 올림픽 때 처음으로 참가했다. 그 전에는 소...,Olympic Sports Historian,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer


In [33]:
row_idx = 0
q = eval_df.loc[row_idx, 'user_input'] # 질문 조회
resp = chain.invoke(q) # dict[response, retrieved_context]



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Country {id: "프랑스"})-[:HOST]->(e:Event)-[:HELD_IN]->(s:Season {id: "동계"}), (e)-[:HELD_IN]->(d:Date) 
RETURN e.id AS OlympicEvent, d.id AS HeldDate
Full Context:
[]

> Finished chain.


In [34]:
resp['response']

'모르겠습니다.'

In [35]:
resp['retrieved_context']

[]

In [37]:
# testset의 모든 데이터에 llm 응답과 retriever의 검색결과를 추가.
response_list = [] # LLM 응답들을 저장할 리스트
retrieved_context_list = [] # retriever가 검색한 문서들을 저장할 리스트

for user_input in eval_df['user_input']:
    resp = chain.invoke(user_input)
    response_list.append(resp['response'])
    retrieved_context_list.append(resp['retrieved_context'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Country {id:"프랑스"})<-[:HELD_IN]-(e:Event)-[:HELD_IN]->(d:Date)
RETURN e.id AS event, d.id AS 개최일
Full Context:
[{'event': '1회 동계올림픽', '개최일': '1924년'}]

> Finished chain.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (e:Event)-[:START_DATE]->(d:Date {id: "1948"}), (e)-[:수여]->(r:Rank)
RETURN r.id
Full Context:
[]

> Finished chain.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH
  (helsinki:Event)<-[:MENTIONS]-(doc:Document),
  (ussr:Country {id:"소련"}),
  (ussr)-[:처음_참가]->(helsinki),
  OPTIONAL MATCH (ussr)-[:참가]->(e:Event)
WITH helsinki, ussr, collect(e) AS events
OPTIONAL MATCH (ussr)-[:처음_참가]->(helsinki)<-[:MENTIONS]-(background_doc:Document)
OPTIONAL MATCH (ussr)-[:우승]->(olympic_event:Event)
RETURN helsinki.id AS helsinki_olympics,
       ussr.id AS country,
       background_doc.text AS participation_background,
       [event IN events | event.id] AS olympic_participat

CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'MATCH': expected a graph pattern (line 5, column 12 (offset: 124))
"  OPTIONAL MATCH (ussr)-[:참가]->(e:Event)"
            ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}

In [38]:
print(len(response_list), len(retrieved_context_list))

2 2


In [39]:
response_list

['프랑스에서 열린 동계 올림픽은 1924년에 개최되었습니다.', '알려진 정보가 없어 답변할 수 없습니다.']

In [40]:
retrieved_context_list

[["{'event': '1회 동계올림픽', '개최일': '1924년'}"], []]

In [41]:
######################################
# eval_df 에 컬럼으로 추가
######################################
eval_df['response'] = response_list
eval_df['retrieved_contexts'] = retrieved_context_list
eval_df.head()

ValueError: Length of values (2) does not match length of index (10)

In [42]:
# eval_df 를 RAGAS의 평가데이터셋 타입으로 변환. 
from ragas import EvaluationDataset
eval_dataset = EvaluationDataset.from_pandas(
    eval_df[["user_input", "retrieved_contexts", "response", "reference"]]
)
eval_dataset

KeyError: "['retrieved_contexts', 'response'] not in index"

In [43]:
####################
# 평가
####################
from ragas.metrics import  (
    LLMContextRecall, # Context Recall
    LLMContextPrecisionWithReference, # Context Precision
    Faithfulness,
    AnswerRelevancy
)
from ragas import evaluate 

C:\Users\Playdata\AppData\Local\Temp\ipykernel_8700\858032146.py:4: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import  (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_8700\858032146.py:4: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import  (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_8700\858032146.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import  (
C:\Users\P

In [44]:
# 평가할 때 사용할 LLM, Embedding모델
eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

# Metric(평가지표) 객체를 List로 묶어준다.
## 내가 평가할 지표들만 묶어준다.
metrics = [
    LLMContextRecall(llm=eval_llm),
    LLMContextPrecisionWithReference(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings=eval_embeddings)
]
# 평가진행
eval_result = evaluate(dataset=eval_dataset, metrics=metrics)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_8700\2786112854.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_8700\2786112854.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


NameError: name 'eval_dataset' is not defined

In [45]:
eval_result

NameError: name 'eval_result' is not defined

In [46]:
# print(type(eval_result))
## 개별 평가데이터에 대한 평가점수.
result_df = eval_result.to_pandas()

NameError: name 'eval_result' is not defined

In [47]:
result_df

NameError: name 'result_df' is not defined